# Get to Know a Dataset: [Version 2 High Resolution Canopy Height Maps by WRI and Meta]

This notebook serves as a guided tour of the [Version 2 High Resolution Canopy Height Maps by WRI and Meta](https://registry.opendata.aws/dataforgood-fb-forestsv2/) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).



### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.



At the top level of our S3 bucket ("dataforgood-fb-data"), we have a prefix "forests/v2/global/dinov3_global_chm_v2_ml3"  contains:

 1. "chm" containing canopy height maps as cloud optimized geotiffs.
 2. "metadata" containing geojsons with observation date across the dataset.
 3. "tiles.geojson" is a geojson containing the tile extent for each tile, and the associated quadkey name.
 
 Full documentation for this dataset can be found at: https://arxiv.org/abs/2603.06382





First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# matplotlib >= 3.10.3 
# rasterio >= 1.5.0

# Import the libraries required for this notebook
# Built-ins
import json
from pprint import pprint
import tempfile
import os
# Installed libraries
import boto3, matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config
import rasterio


Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level prefixes in our S3 path:


In [ ]:
# Location of the S3 bucket for this dataset
bucket = "dataforgood-fb-data"
path = "forests/v2/global/dinov3_global_chm_v2_ml3/"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
for item in s3.list_objects_v2(Bucket=bucket, Prefix=path, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])




Looking into the geotiff prefix of our dataset, we see a list of .tif files, with names cooresponding to quadkey tiles at zoom_level=10.


In [ ]:
path = "forests/v2/global/dinov3_global_chm_v2_ml3/"


# each page has a max of 1000 items
paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket=bucket, Prefix=path)

outlist = []
#only print first page here
for page in pages:
    if "Contents" in page.keys():
        objlist = [i["Key"] for i in page["Contents"]]
        outlist.extend(objlist)
        break
#we only print 10 here
pprint(outlist[-10:])    



### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?


Our dataset comes as a set of Cloud Optimized Geotiffs:

-  The extent of each geotiff is a zoom_level=10 [web mercator tile](https://en.wikipedia.org/wiki/Web_Mercator_projection).
-  The filenames are quadkeys of the containing tile.
-  Each geotiff contains a single data band, which represents the top of canopy height above the ground in meters.
-  The mask band of the geotiff is a boolean represnting where or not the input imagery has been flagged as containing a cloud.
-  The CRS is epsg:3857


The geojsons contain a set of polygons in a given tile. 
- Each polygon contains a single feature value, containing a string of the observation date of the input imagery. 


### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, let us load up and look at one geotiff


In [ ]:
#download chm
s3file="forests/v2/global/dinov3_global_chm_v2_ml3/chm/0022222122.tif"
with tempfile.NamedTemporaryFile(suffix=".tif") as dst:
    s3.download_file(bucket, s3file, dst.name)
    with rasterio.open(dst.name) as src:
        chm=src.read().squeeze()
        meta=src.meta
print(meta)

In [ ]:
plt.imshow(chm[0:1000,0:1000])



### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

We have used the data to identify relative canopy height of two nearby areas. When evaluating forest restoration and carbon stroage potential, it is useful to compare the existing state of canopy volume (ie, integrated canopy height) for a gien area, compared to the canopy valume in a mature forest nearby. 

This example highlight the strengths of the dataset (high resolution canopy height estimates, available globally), while minimizing some weaknesses (errors related to view angle, data available from a single time) by making relative (rather than absolute) measurements.




### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?


The connection between canopy height maps and biomass is a challenging but important link for carbon markets. Solving this problem would be valuable for not just this type of dataset, but aerial lidar datasets as well.

